In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

class TimeSeriesModel:
    """
    A class to perform simple Time Series Analysis and Forecasting.
    Uses a Simple Moving Average and basic Autoregressive logic.
    """
    
    def __init__(self, window_size=5):
        self.window_size = window_size
        self.last_window = None
        self.slope = None
        self.intercept = None

    def fit_autoregressive(self, data):
        """
        Fits a simple AR(1) style model using OLS: y_t = B0 + B1 * y_{t-1}
        """
        # Create lagged features
        y = data[1:]
        X = data[:-1].reshape(-1, 1)
        
        # Add intercept column
        X_b = np.c_[np.ones(X.shape[0]), X]
        
        # Solve using Normal Equation: (X^T * X)^-1 * X^T * y
        theta = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
        
        self.intercept = theta[0]
        self.slope = theta[1]
        self.last_window = data[-self.window_size:]
        
        print(f"--- Model Training Complete ---")
        print(f"Trend Coefficient: {self.slope:.4f}")
        print(f"Base Level (Intercept): {self.intercept:.4f}")

    def predict_next(self, steps=5):
        """
        Forecasts future values based on the last observed window and trend.
        """
        predictions = []
        current_val = self.last_window[-1]
        
        for _ in range(steps):
            next_val = self.intercept + self.slope * current_val
            predictions.append(next_val)
            current_val = next_val
            
        return np.array(predictions)

def load_timeseries_csv(file_path, date_col, value_col):
    """
    Loads a CSV, parses dates, and prepares the series for analysis.
    """
    print(f"Loading data from {file_path}...")
    df = pd.read_csv(file_path)
    
    # 1. Convert date column to datetime objects
    df[date_col] = pd.to_datetime(df[date_col])
    
    # 2. Sort by date (essential for Time Series)
    df = df.sort_values(by=date_col)
    
    # 3. Handle missing values (Interpolation is best for time series)
    df[value_col] = df[value_col].interpolate(method='linear')
    
    return df[date_col].values, df[value_col].values

# --- CONFIGURATION SECTION ---
# Edit these variables to adapt the script to your dataset
CONFIG = {
    "file_path": "your_time_series.csv", # CHANGE 1: Path to your CSV
    "date_column": "Date",                # CHANGE 2: Name of your timestamp column
    "value_column": "Close",              # CHANGE 3: Column you want to predict (e.g., Price, Sales)
    "forecast_steps": 10,                 # CHANGE 4: Number of points to predict into the future
    "window_size": 7                      # CHANGE 5: Look-back window for trends
}

if __name__ == "__main__":
    try:
        # 1. Load and Prepare
        dates, values = load_timeseries_csv(
            CONFIG["file_path"], 
            CONFIG["date_column"], 
            CONFIG["value_column"]
        )

        # 2. Initialize and Train
        model = TimeSeriesModel(window_size=CONFIG["window_size"])
        model.fit_autoregressive(values)

        # 3. Forecast
        future_predictions = model.predict_next(steps=CONFIG["forecast_steps"])
        
        # Generate future dates for plotting
        date_diff = dates[1] - dates[0]
        future_dates = [dates[-1] + (i + 1) * date_diff for i in range(CONFIG["forecast_steps"])]

        print(f"\nForecast for next {CONFIG['forecast_steps']} periods:")
        print(future_predictions)

        # 4. Visualization
        plt.figure(figsize=(12, 6))
        plt.plot(dates, values, label='Historical Data', color='blue', linewidth=1.5)
        plt.plot(future_dates, future_predictions, 'o--', label='Forecast', color='red', markersize=4)
        
        plt.title(f'Time Series Forecast for {CONFIG["value_column"]}')
        plt.xlabel('Time')
        plt.ylabel(CONFIG["value_column"])
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.show()

    except FileNotFoundError:
        print(f"Error: Could not find file '{CONFIG['file_path']}'.")
    except Exception as e:
        print(f"An error occurred: {e}")